# 01 Build the merged dataset (run once)
Merges the raw TALIS 2024 teacher file (`ttgintt4.sav`) with the principal file (`tcgintt4.sav`) into one row per teacher, and writes `Data/output/teacher_principal_named_columns.csv` (~1.6 GB).

**Slow (raw files are several hundred MB). You only ever need this once.** If you already have the merged CSV (e.g. downloaded via the link in `Data/README.md`), place it in `Data/output/` and go straight to `02_modeling.ipynb`.

Requires the raw `.sav` files in `Data/SPSS/` — download instructions in `Data/README.md`.

In [1]:
# ============================================================
# CELL 0 — paths (portable: finds the repo by walking up from cwd)
# No editing needed on any machine. If it errors, open the repo
# folder itself in VS Code / Jupyter and restart the kernel.
# ============================================================
from pathlib import Path

def find_root(start=None, depth=6):
    p = start or Path.cwd()
    for _ in range(depth):
        if (p / "Data").exists() and (p / "Model").exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        f"repo root not found walking up from {Path.cwd()} — "
        "in VS Code use File > Open Folder on summer26-teacher-ai-readiness, "
        "reopen this notebook, restart the kernel")

ROOT = find_root()
DATA_DIR = ROOT / "Data"                 # codebook + small CSVs
SPSS_DIR = DATA_DIR / "SPSS"             # raw TALIS .sav files (gitignored)
OUT_DIR  = DATA_DIR / "output"           # everything the notebooks produce (gitignored)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)

repo root: c:\Users\elif_\Documents\summer26-teacher-ai-readiness


In [2]:
# ============================================================
# CELL 1 — BUILD merged file from raw .sav (RUN ONCE / on first setup)
# Skip this whole notebook if Data/output/teacher_principal_named_columns.csv
# already exists — 02_modeling loads it directly.
# ============================================================

import pyreadstat

# user_missing=True is what keeps 8/9, 98/99, 998/999 etc. as actual values
# instead of converting them to NaN (blank in the CSV). Keep it on BOTH reads.
tdf, tmeta = pyreadstat.read_sav(
    SPSS_DIR / "ttgintt4.sav", user_missing=True)
pdf, pmeta = pyreadstat.read_sav(
    SPSS_DIR / "tcgintt4.sav", user_missing=True)

# --- decide the join keys (check if schools repeat across ISCED populations) ---
multi_pop = pdf.groupby(["IDCNTRY", "IDSCHOOL"])["IDPOP"].nunique().gt(1).sum()
keys = ["IDCNTRY", "IDPOP", "IDSCHOOL"] if multi_pop > 0 else ["IDCNTRY", "IDSCHOOL"]
print("join keys:", keys)

# --- drop only true admin redundancies; ADJRT24 now stays ---
admin_dupes = ["CNTRY", "IDCNTPOP", "VERSION", "IEADATE"]
if "IDPOP" not in keys:
    admin_dupes.append("IDPOP")          # only redundant if it's not a key
pdf_slim = pdf.drop(columns=[c for c in admin_dupes if c in pdf.columns])

# --- P_ prefix on every principal column except the keys ---
pdf_slim = pdf_slim.rename(
    columns={c: f"P_{c}" for c in pdf_slim.columns if c not in keys})

# --- merge: one row per teacher, principal/school answers attached ---
merged = tdf.merge(pdf_slim, on=keys, how="left", validate="m:1")

# --- rename to "CODE question", stripping P_ for the label lookup ---
labels = {**tmeta.column_names_to_labels, **pmeta.column_names_to_labels}

def label_for(col):
    base = col[2:] if col.startswith("P_") else col
    lab = labels.get(base)
    return f"{col} {lab}" if lab else col

merged = merged.rename(columns={c: label_for(c) for c in merged.columns})

# NOTE: now writes to Data/output/ — v5 wrote to Data/ root while CELL 1 of the
# modeling notebook read from Data/output/; this aligns the two.
merged.to_csv(OUT_DIR / "teacher_principal_named_columns.csv",
              index=False, encoding="utf-8-sig")

# --- sanity check: join success (uses the P_ prefix now) ---
pcols = [c for c in merged.columns if c.startswith("P_TC")]
matched = merged[pcols].notna().any(axis=1)
print("teachers:", len(merged))
print("matched to a principal:", matched.sum(), f"({matched.mean()*100:.1f}%)")

join keys: ['IDCNTRY', 'IDSCHOOL']
teachers: 278383
matched to a principal: 269110 (96.7%)
